# Reproduce R24F_MIXED_lr0.5_V08 — TRUE CHAMPION baseline

Riproduce il run R24F originale (PROVED CLEAN: gn_max=21.79, val_data=0.181).

**Importante**: questo notebook va eseguito **dalla root del repo** (`CF_FSNN/`), NON dalla cartella `Arch_Tested/...`. Il train.py corrente è già backward-compatibile (R29 fixes sono opt-in via flag).

Cosa fa:
- Cell 1: smoke 1ep × 3step → verifica CLI flow + `gn_max < 25`
- Cell 2: full 10ep replica → val_data ≈ 0.181 (match snapshot ± seed drift)
- Cell 3: confronto con snapshot_original/training_log.csv

Output: `checkpoints/R24F_baseline_replica_v2/` con CSV + plots + best_model.pt

In [ ]:
# Cell 1 -- Smoke 1ep × 3step (verifica CLI flow + sanity gn)
import sys, os, subprocess, time, json, shutil

# Esecuzione da CF_FSNN/ root (non dalla cartella Arch_Tested)
ROOT = os.getcwd()
assert os.path.isfile('train.py'), f'Lancia da CF_FSNN/ root, non da {ROOT}'
print(f'CWD: {ROOT}')

tag_smoke = f'_R24F_REPLICA_SMOKE_{int(time.time())}'
cmd = [sys.executable, 'train.py',
    '--training_method', 'baseline',
    '--epochs', '1', '--max_steps_per_epoch', '3',
    '--batch_size', '8', '--val_batch_size', '32',
    '--seq_len', '50',
    '--cf_hidden_size', '32', '--cf_rank', '8',
    '--cf_max_delay', '6', '--cf_bit_shift', '3',
    # NESSUN R29 flag → opzioni default = no-op (backward-compat)
    '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
    '--lambda_bc', '1.0', '--lambda_sr', '0.5', '--lambda_T_aux', '0.0',
    '--scenario_mix', 'highway:0.4,urban:0.3,truck:0.2,mixed:0.1',
    '--cut_in_ratio', '0.0', '--noise_scale', '0.0',
    '--n_train', '80', '--n_val', '40',
    '--optimizer', 'prodigy', '--lr', '0.5', '--max_lr', '0.5',
    '--scheduler', 'cosine_no_restart',
    '--prodigy_betas', '0.9,0.99', '--prodigy_d_coef', '1.0',
    '--prodigy_d0', '1e-6', '--prodigy_weight_decay', '0.01',
    '--prodigy_use_bias_correction', '1', '--prodigy_safeguard_warmup', '1',
    '--prodigy_growth_rate', 'inf',
    '--max_inf_streak', '99999', '--early_stop_patience', '0',
    '--tag', tag_smoke]

print(f'\nSmoke: lr=0.5 (NOT 1.0), seq_len=50, baseline, 1ep × 3step')
r = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8', errors='replace')
if r.returncode != 0:
    print('STDOUT:', r.stdout[-1500:])
    print('STDERR:', r.stderr[-500:])
    raise RuntimeError(f'smoke failed (exit {r.returncode})')

# Sanity check on gn_total_preclip
import pandas as pd, math
bdf = pd.read_csv(f'checkpoints/{tag_smoke}/training_batch_log.csv')
gn = bdf['gn_total_preclip']
gn_finite = gn[gn.apply(math.isfinite)]
print(f'\nSmoke gn_total_preclip: mean={gn_finite.mean():.3f}  max={gn_finite.max():.3f}')
print(f'inf grads: {(~gn.apply(math.isfinite)).sum()}')

# Verifica R29 flags NON attivi (backward-compat)
cfg = json.load(open(f'checkpoints/{tag_smoke}/config_snapshot.json'))
assert cfg.get('cf_init_bias_shift', 0) == 0, 'init_bias_shift dovrebbe essere 0 di default'
assert cfg.get('cf_logit_tau_init', 1.0) == 1.0, 'logit_tau_init dovrebbe essere 1.0 di default'
print('  [OK] R29 flags non attivi (default = no-op)')
print('  [OK] lr=0.5 attivo')

# Cleanup smoke
def _robust_rmtree(path, max_retries=3):
    for attempt in range(max_retries):
        if not os.path.isdir(path): return True
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path): return True
        time.sleep(0.5 * (attempt + 1))
    return not os.path.isdir(path)
_robust_rmtree(f'checkpoints/{tag_smoke}')
print('\nSMOKE PASSED.')

In [ ]:
# Cell 2 -- Full 10ep replica (val_data atteso ≈ 0.181)
# Esegui solo se vuoi riprodurre il vero baseline (~10-15 min su Azure CPU).
import sys, subprocess, time

TAG = 'R24F_baseline_replica_v2'   # output: checkpoints/<TAG>/

cmd = [sys.executable, 'train.py',
    '--training_method', 'baseline',
    '--epochs', '10', '--max_steps_per_epoch', '100',
    '--batch_size', '8', '--val_batch_size', '32',
    '--seq_len', '50',
    '--cf_hidden_size', '32', '--cf_rank', '8',
    '--cf_max_delay', '6', '--cf_bit_shift', '3',
    '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
    '--lambda_bc', '1.0', '--lambda_sr', '0.5', '--lambda_T_aux', '0.0',
    '--scenario_mix', 'highway:0.4,urban:0.3,truck:0.2,mixed:0.1',
    '--cut_in_ratio', '0.0', '--noise_scale', '0.0',
    '--n_train', '1500', '--n_val', '300',
    '--optimizer', 'prodigy', '--lr', '0.5', '--max_lr', '0.5',
    '--scheduler', 'cosine_no_restart',
    '--prodigy_betas', '0.9,0.99', '--prodigy_d_coef', '1.0',
    '--prodigy_d0', '1e-6', '--prodigy_weight_decay', '0.01',
    '--prodigy_use_bias_correction', '1', '--prodigy_safeguard_warmup', '1',
    '--prodigy_growth_rate', 'inf',
    '--max_inf_streak', '99999', '--early_stop_patience', '0',
    '--data_cache', 'data/cache_1500_mixed_cut0.0_ou0.0.pt',
    '--tag', TAG]

print(f'Full replica: lr=0.5, seq_len=50, 10ep × 100step, mixed scenario')
print(f'Output → checkpoints/{TAG}/')
t0 = time.time()
r = subprocess.run(cmd, capture_output=False)   # stream stdout
el = time.time() - t0
print(f'\nReplica done in {el/60:.1f} min, exit={r.returncode}')

In [ ]:
# Cell 3 -- Confronto vs snapshot_original
import os, math
import pandas as pd

# Path relativi alla root CF_FSNN/
SNAP = 'Arch_Tested/R24F_MIXED_lr0.5_V08_TRUE_CHAMPION/snapshot_original'
REPLICA = f'checkpoints/R24F_baseline_replica_v2'

if not os.path.isfile(f'{REPLICA}/training_log.csv'):
    print(f'⚠ Replica non eseguita ({REPLICA}/training_log.csv mancante). Esegui Cell 2 prima.')
else:
    orig = pd.read_csv(f'{SNAP}/training_log.csv')
    new  = pd.read_csv(f'{REPLICA}/training_log.csv')
    print(f'Confronto orig vs replica:')
    print(f'  orig val_data per ep:    {[f"{v:.4f}" for v in orig.val_data.tolist()]}')
    print(f'  replica val_data per ep: {[f"{v:.4f}" for v in new.val_data.tolist()]}')
    print(f'\n  orig val_data min: {orig.val_data.min():.4f}  (ep {orig.val_data.idxmin()+1})')
    print(f'  new  val_data min: {new.val_data.min():.4f}  (ep {new.val_data.idxmin()+1})')
    drift = new.val_data.min() - orig.val_data.min()
    pct = drift / orig.val_data.min() * 100
    print(f'  drift: {drift:+.4f} ({pct:+.1f}%)')

    if 'val_T_intra_corr' in new.columns:
        print(f'\n  [NEW R27] val_T_intra_corr per ep (replica):')
        print(f'    {[f"{v:.3f}" for v in new.val_T_intra_corr.tolist()]}')

    # Sanity gn_max
    bdf = pd.read_csv(f'{REPLICA}/training_batch_log.csv')
    gn = bdf['gn_total_preclip']
    gn_f = gn[gn.apply(math.isfinite)]
    print(f'\n  Replica gn_total_preclip: mean={gn_f.mean():.3f}, max={gn_f.max():.3f}')
    print(f'  Original gn max=21.79; replica max should be < 25 for clean training')
    if gn_f.max() < 25:
        print(f'  ✅ CLEAN: gradienti sotto controllo')
    else:
        print(f'  ⚠ Drift di gradiente — investigare')